# EagleVision on SCARED / Endoscopy Stereo

This Kaggle notebook runs the controlled SCARED comparison:

1. Frozen Depth Anything V2-Small
2. Frozen DA2-Small + EagleVision tiny output residual log-depth adapter
3. DA2-Small + DARES-style internal LoRA baseline

You can either attach a SCARED-style folder under `/kaggle/input`, or load the Hugging Face dataset `juseonghan/SCARED-C` with `datasets.load_dataset`. The Hugging Face path exports decoded samples into an EagleVision manifest-friendly layout before training.


In [ ]:
from pathlib import Path
import os

# ---- Edit these for your Kaggle run ----
REPO_DIR = Path(os.environ.get("EAGLEVISION_REPO", "/kaggle/working/EagleVision"))

# Option A: Hugging Face SCARED-C. This is enabled by default for this notebook.
# If the dataset is gated/private, add an HF_TOKEN Kaggle secret or run huggingface-cli login.
USE_HF_SCARED = os.environ.get("USE_HF_SCARED", "1") == "1"
SCARED_HF_DATASET = os.environ.get("SCARED_HF_DATASET", "juseonghan/SCARED-C")
HF_EXPORT_ROOT = Path(os.environ.get("SCARED_HF_EXPORT_ROOT", "/kaggle/working/scared_hf_export"))

# Option B: already-mounted SCARED folder. Used when USE_HF_SCARED=False.
SCARED_ROOT = Path(os.environ.get("SCARED_ROOT", "/kaggle/input/SCARED"))

# Optional: set this if automatic left/right pairing cannot infer your SCARED layout.
# CSV columns: sample_id, sequence_id, frame_id, left_rgb, right_rgb, depth, disparity,
# mask, K_left, K_right, T_left_to_right, baseline, focal_length, split
MAPPING_CSV = os.environ.get("SCARED_MAPPING_CSV", "")

# Fallback geometry if the HF rows do not expose per-sample calibration.
# Replace these with the calibration values for your SCARED-C export when available.
HF_DEFAULT_BASELINE = float(os.environ.get("SCARED_BASELINE", "0.005"))
HF_DEFAULT_FOCAL_LENGTH = float(os.environ.get("SCARED_FOCAL_LENGTH", "700.0"))

# Optional DA2 checkpoint. Leave as None to use whatever your local baseline config provides.
DA2_CHECKPOINT = os.environ.get("DA2_CHECKPOINT", "") or None

IMAGE_SIZE = [256, 320]
EPOCHS = 5
BATCH_SIZE = 4
NUM_WORKERS = 2

# Set QUICK_RUN=True for a smoke test before a real Kaggle run.
QUICK_RUN = False
MAX_STEPS_PER_EPOCH = 2 if QUICK_RUN else None

RUN_FROZEN_EVAL = True
RUN_ADAPTER_DRY_RUN = True
RUN_ADAPTER_TRAIN = True
RUN_LORA_TRAIN = True

if not REPO_DIR.exists() and Path.cwd().name == "EagleVision":
    REPO_DIR = Path.cwd()

print("Repo:", REPO_DIR)
print("Use HF SCARED-C:", USE_HF_SCARED)
print("HF dataset:", SCARED_HF_DATASET)
print("SCARED root:", SCARED_ROOT)
print("Mapping CSV:", MAPPING_CSV or "<auto discovery or HF export>")
print("DA2 checkpoint:", DA2_CHECKPOINT or "<config/default>")
print("Quick run:", QUICK_RUN)


In [ ]:
import os, sys, subprocess, json, textwrap
from pathlib import Path

assert REPO_DIR.exists(), f"Repo not found: {REPO_DIR}"
os.chdir(REPO_DIR)
print("cwd:", Path.cwd())

# Install EagleVision as an editable package. Kaggle usually already has torch installed.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

## Optional: Load SCARED-C From Hugging Face

This cell implements:

```python
from datasets import load_dataset
ds = load_dataset("juseonghan/SCARED-C")
```

It then exports images/arrays to local files and writes a mapping CSV for EagleVision. If your Hugging Face account needs access, add `HF_TOKEN` as a Kaggle secret.


In [ ]:
if USE_HF_SCARED:
    import csv, shutil
    import numpy as np
    from PIL import Image

    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        HF_TOKEN = os.environ.get("HF_TOKEN")

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets", "huggingface_hub"], check=True)
    from datasets import load_dataset

    print(f"Loading Hugging Face dataset: {SCARED_HF_DATASET}")
    try:
        ds = load_dataset(SCARED_HF_DATASET, token=HF_TOKEN)
    except TypeError:
        ds = load_dataset(SCARED_HF_DATASET, use_auth_token=HF_TOKEN)

    print(ds)
    HF_EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
    mapping_path = REPO_DIR / "manifests" / "scared" / "scared_hf_mapping.csv"
    mapping_path.parent.mkdir(parents=True, exist_ok=True)

    LEFT_CANDIDATES = ["left_rgb", "left", "image_left", "left_image", "left_img", "img_left", "image_l", "rgb_left"]
    RIGHT_CANDIDATES = ["right_rgb", "right", "image_right", "right_image", "right_img", "img_right", "image_r", "rgb_right"]
    DEPTH_CANDIDATES = ["depth", "depth_gt", "left_depth", "depth_left"]
    DISP_CANDIDATES = ["disparity", "disp", "disp_gt", "left_disparity", "disparity_left"]
    MASK_CANDIDATES = ["mask", "valid_mask", "valid", "left_mask"]
    BASELINE_CANDIDATES = ["baseline", "baseline_m", "stereo_baseline"]
    FOCAL_CANDIDATES = ["focal_length", "focal", "fx", "focal_left"]

    def choose_col(columns, candidates):
        lower = {c.lower(): c for c in columns}
        for cand in candidates:
            if cand.lower() in lower:
                return lower[cand.lower()]
        for col in columns:
            low = col.lower()
            if any(cand.lower() in low for cand in candidates):
                return col
        return None

    def save_value(value, path: Path, prefer_image=False):
        path.parent.mkdir(parents=True, exist_ok=True)
        if value is None:
            return None
        wants_array = path.suffix.lower() == ".npy"
        if isinstance(value, str):
            src = Path(value)
            if src.exists():
                if wants_array and src.suffix.lower() != ".npy":
                    arr = np.asarray(Image.open(src))
                    np.save(path, arr.astype(np.float32))
                    return path
                shutil.copy(src, path)
                return path
            return None
        if isinstance(value, dict):
            if value.get("path") and Path(value["path"]).exists():
                src = Path(value["path"])
                if wants_array and src.suffix.lower() != ".npy":
                    arr = np.asarray(Image.open(src))
                    np.save(path, arr.astype(np.float32))
                    return path
                shutil.copy(src, path)
                return path
            if value.get("bytes") is not None:
                path.write_bytes(value["bytes"])
                return path
        if isinstance(value, Image.Image):
            if wants_array:
                np.save(path, np.asarray(value).astype(np.float32))
                return path
            value.convert("RGB" if prefer_image else value.mode).save(path)
            return path
        array = np.asarray(value)
        if wants_array:
            np.save(path, array.astype(np.float32))
            return path
        if prefer_image or (array.ndim in (2, 3) and array.dtype == np.uint8):
            Image.fromarray(array).save(path)
            return path
        np.save(path.with_suffix(".npy"), array.astype(np.float32))
        return path.with_suffix(".npy")

    def scalar_from(row, candidates, default):
        col = choose_col(row.keys(), candidates)
        if col is None or row.get(col) is None:
            return default
        value = row[col]
        try:
            arr = np.asarray(value)
            return float(arr.reshape(-1)[0])
        except Exception:
            return default

    split_name_map = {"validation": "val", "valid": "val", "test": "test", "train": "train"}
    fieldnames = [
        "sample_id", "sequence_id", "frame_id", "left_rgb", "right_rgb", "depth", "disparity", "mask",
        "K_left", "K_right", "T_left_to_right", "baseline", "focal_length", "split",
    ]
    written = 0
    dataset_items = ds.items() if hasattr(ds, "items") else [("train", ds)]
    with mapping_path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        for split_name, split_ds in dataset_items:
            split = split_name_map.get(split_name.lower(), split_name.lower())
            columns = split_ds.column_names
            left_col = choose_col(columns, LEFT_CANDIDATES)
            right_col = choose_col(columns, RIGHT_CANDIDATES)
            depth_col = choose_col(columns, DEPTH_CANDIDATES)
            disp_col = choose_col(columns, DISP_CANDIDATES)
            mask_col = choose_col(columns, MASK_CANDIDATES)
            if left_col is None or right_col is None:
                raise RuntimeError(
                    f"Could not infer left/right columns for split {split_name}. Columns: {columns}. "
                    "Set SCARED_MAPPING_CSV manually or edit the candidate lists in this cell."
                )
            print(split_name, "columns:", columns)
            print("using:", {"left": left_col, "right": right_col, "depth": depth_col, "disparity": disp_col, "mask": mask_col})
            for idx, row in enumerate(split_ds):
                sample_id = str(row.get("sample_id") or row.get("id") or f"{split}_{idx:06d}")
                sequence_id = str(row.get("sequence_id") or row.get("sequence") or split)
                frame_id = str(row.get("frame_id") or row.get("frame") or idx)
                sample_dir = HF_EXPORT_ROOT / split / sequence_id / sample_id
                left_path = save_value(row[left_col], sample_dir / "left.png", prefer_image=True)
                right_path = save_value(row[right_col], sample_dir / "right.png", prefer_image=True)
                depth_path = save_value(row[depth_col], sample_dir / "depth.npy") if depth_col else None
                disp_path = save_value(row[disp_col], sample_dir / "disparity.npy") if disp_col else None
                mask_path = save_value(row[mask_col], sample_dir / "mask.png", prefer_image=True) if mask_col else None
                writer.writerow({
                    "sample_id": sample_id,
                    "sequence_id": sequence_id,
                    "frame_id": frame_id,
                    "left_rgb": str(left_path.relative_to(HF_EXPORT_ROOT)),
                    "right_rgb": str(right_path.relative_to(HF_EXPORT_ROOT)),
                    "depth": str(depth_path.relative_to(HF_EXPORT_ROOT)) if depth_path else "",
                    "disparity": str(disp_path.relative_to(HF_EXPORT_ROOT)) if disp_path else "",
                    "mask": str(mask_path.relative_to(HF_EXPORT_ROOT)) if mask_path else "",
                    "baseline": scalar_from(row, BASELINE_CANDIDATES, HF_DEFAULT_BASELINE),
                    "focal_length": scalar_from(row, FOCAL_CANDIDATES, HF_DEFAULT_FOCAL_LENGTH),
                    "split": split,
                })
                written += 1

    SCARED_ROOT = HF_EXPORT_ROOT
    MAPPING_CSV = str(mapping_path)
    print(f"Exported {written} Hugging Face samples to {HF_EXPORT_ROOT}")
    print(f"Mapping CSV: {MAPPING_CSV}")
else:
    print("Skipping Hugging Face SCARED-C export; using mounted SCARED_ROOT/MAPPING_CSV.")


In [ ]:
import yaml
from copy import deepcopy

CONFIG_DIR = Path("configs/endo")
MANIFEST_DIR = Path("manifests/scared")
RAW_RESULTS_DIR = Path("results/raw")
AGG_DIR = Path("results/aggregated/scared_head_to_head")

for path in [CONFIG_DIR, MANIFEST_DIR, RAW_RESULTS_DIR, AGG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

def load_yaml(path):
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)

def save_yaml(path, payload):
    with open(path, "w", encoding="utf-8") as f:
        yaml.safe_dump(payload, f, sort_keys=False)
    print("wrote", path)

def kaggleize(cfg, output_dir=None):
    cfg = deepcopy(cfg)
    cfg["device"] = "cuda" if torch.cuda.is_available() else "cpu"
    cfg.setdefault("data", {})
    cfg["data"]["root"] = str(SCARED_ROOT)
    cfg["data"]["image_size"] = IMAGE_SIZE
    cfg.setdefault("base_model", {})
    if DA2_CHECKPOINT:
        cfg["base_model"]["checkpoint_path"] = DA2_CHECKPOINT
    if "train" in cfg:
        cfg["train"]["epochs"] = EPOCHS
        cfg["train"]["batch_size"] = BATCH_SIZE
        cfg["train"]["num_workers"] = NUM_WORKERS
        cfg["train"]["max_steps_per_epoch"] = MAX_STEPS_PER_EPOCH
    cfg.setdefault("eval", {})
    cfg["eval"]["batch_size"] = 1
    cfg["eval"]["num_workers"] = NUM_WORKERS
    if output_dir is not None:
        cfg["output_dir"] = output_dir
    return cfg

frozen_cfg = kaggleize(load_yaml(CONFIG_DIR / "frozen_da2s.yaml"))
adapter_cfg = kaggleize(load_yaml(CONFIG_DIR / "eaglevision_da2s_cycle.yaml"), "outputs/endo/eaglevision_da2s_cycle")
lora_cfg = kaggleize(load_yaml(CONFIG_DIR / "lora_da2s_cycle_r4.yaml"), "outputs/endo/lora_da2s_cycle_r4")

save_yaml(CONFIG_DIR / "kaggle_frozen_da2s.yaml", frozen_cfg)
save_yaml(CONFIG_DIR / "kaggle_eaglevision_da2s_cycle.yaml", adapter_cfg)
save_yaml(CONFIG_DIR / "kaggle_lora_da2s_cycle_r4.yaml", lora_cfg)

## Build And Validate Manifests

Automatic discovery works only for common left/right folder naming. If this cell fails, create a mapping CSV and set `MAPPING_CSV` in the first cell.

In [ ]:
cmd = [
    sys.executable, "-m", "eaglevision.cli.build_scared_manifest",
    "--root", str(SCARED_ROOT),
    "--out-dir", str(MANIFEST_DIR),
    "--seed", "42",
]
if MAPPING_CSV:
    cmd.extend(["--mapping-csv", MAPPING_CSV])
print(" ".join(cmd))
subprocess.run(cmd, check=True)

for split in ["train", "val", "test"]:
    manifest = MANIFEST_DIR / f"{split}.jsonl"
    out = MANIFEST_DIR / f"{split}_validation.json"
    cmd = [
        sys.executable, "-m", "eaglevision.cli.validate_endoscopy_manifest",
        "--manifest", str(manifest),
        "--root", str(SCARED_ROOT),
        "--require-geometry",
        "--out", str(out),
    ]
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)
    print(split, json.loads(out.read_text()) | {"errors": "<omitted>"})

## Inspect DA2 Modules For LoRA Targets

In [ ]:
subprocess.run([
    sys.executable, "-m", "eaglevision.cli.inspect_da2_modules",
    "--config", str(CONFIG_DIR / "kaggle_frozen_da2s.yaml"),
    "--contains", "q", "proj", "attn", "linear", "mlp",
    "--out", "outputs/endo/da2_module_inspection.json",
], check=True)

## Frozen DA2-Small Evaluation

In [ ]:
if RUN_FROZEN_EVAL:
    subprocess.run([
        sys.executable, "-m", "eaglevision.cli.eval_endoscopy",
        "--config", str(CONFIG_DIR / "kaggle_frozen_da2s.yaml"),
        "--manifest", str(MANIFEST_DIR / "test.jsonl"),
        "--out", str(RAW_RESULTS_DIR / "frozen_da2s_scared.csv"),
    ], check=True)

## EagleVision Residual Adapter

In [ ]:
if RUN_ADAPTER_DRY_RUN:
    subprocess.run([
        sys.executable, "-m", "eaglevision.cli.train_endoscopy_adapter",
        "--config", str(CONFIG_DIR / "kaggle_eaglevision_da2s_cycle.yaml"),
        "--train-manifest", str(MANIFEST_DIR / "train.jsonl"),
        "--val-manifest", str(MANIFEST_DIR / "val.jsonl"),
        "--dry-run",
    ], check=True)

if RUN_ADAPTER_TRAIN:
    subprocess.run([
        sys.executable, "-m", "eaglevision.cli.train_endoscopy_adapter",
        "--config", str(CONFIG_DIR / "kaggle_eaglevision_da2s_cycle.yaml"),
        "--train-manifest", str(MANIFEST_DIR / "train.jsonl"),
        "--val-manifest", str(MANIFEST_DIR / "val.jsonl"),
    ], check=True)

adapter_ckpt = Path("outputs/endo/eaglevision_da2s_cycle/checkpoints/best.pt")
if adapter_ckpt.exists():
    subprocess.run([
        sys.executable, "-m", "eaglevision.cli.eval_endoscopy",
        "--config", str(CONFIG_DIR / "kaggle_eaglevision_da2s_cycle.yaml"),
        "--manifest", str(MANIFEST_DIR / "test.jsonl"),
        "--checkpoint", str(adapter_ckpt),
        "--out", str(RAW_RESULTS_DIR / "eaglevision_da2s_cycle_scared.csv"),
    ], check=True)
else:
    print("Adapter checkpoint not found; skipping adapter eval:", adapter_ckpt)

## DARES-Style LoRA Baseline

In [ ]:
if RUN_LORA_TRAIN:
    subprocess.run([
        sys.executable, "-m", "eaglevision.cli.train_endoscopy_lora",
        "--config", str(CONFIG_DIR / "kaggle_lora_da2s_cycle_r4.yaml"),
        "--train-manifest", str(MANIFEST_DIR / "train.jsonl"),
        "--val-manifest", str(MANIFEST_DIR / "val.jsonl"),
    ], check=True)

lora_ckpt = Path("outputs/endo/lora_da2s_cycle_r4/checkpoints/best.pt")
if lora_ckpt.exists():
    subprocess.run([
        sys.executable, "-m", "eaglevision.cli.eval_endoscopy",
        "--config", str(CONFIG_DIR / "kaggle_lora_da2s_cycle_r4.yaml"),
        "--manifest", str(MANIFEST_DIR / "test.jsonl"),
        "--checkpoint", str(lora_ckpt),
        "--out", str(RAW_RESULTS_DIR / "lora_da2s_cycle_r4_scared.csv"),
    ], check=True)
else:
    print("LoRA checkpoint not found; skipping LoRA eval:", lora_ckpt)

## Aggregate The Head-To-Head

In [ ]:
inputs = [
    RAW_RESULTS_DIR / "frozen_da2s_scared.csv",
    RAW_RESULTS_DIR / "eaglevision_da2s_cycle_scared.csv",
    RAW_RESULTS_DIR / "lora_da2s_cycle_r4_scared.csv",
]
inputs = [path for path in inputs if path.exists()]
param_summaries = [
    Path("outputs/endo/eaglevision_da2s_cycle/param_summary.json"),
    Path("outputs/endo/lora_da2s_cycle_r4/param_summary.json"),
]
param_summaries = [path for path in param_summaries if path.exists()]

cmd = [
    sys.executable, "scripts/aggregate_endoscopy_results.py",
    "--inputs", *map(str, inputs),
    "--baseline", "frozen_da2s",
    "--out-dir", str(AGG_DIR),
]
if param_summaries:
    cmd.extend(["--param-summaries", *map(str, param_summaries)])
print(" ".join(cmd))
subprocess.run(cmd, check=True)

print((AGG_DIR / "summary.md").read_text())
print((AGG_DIR / "method_ranking.md").read_text()[:4000])

## Preview Results And Panels

In [ ]:
import pandas as pd
from IPython.display import display, Image

summary_csv = AGG_DIR / "summary.csv"
if summary_csv.exists():
    display(pd.read_csv(summary_csv))

for panel in [
    Path("outputs/endo/eaglevision_da2s_cycle/panels/dry_run.png"),
    Path("outputs/endo/eaglevision_da2s_cycle/panels/epoch_001.png"),
    Path("outputs/endo/lora_da2s_cycle_r4/panels/epoch_001.png"),
]:
    if panel.exists():
        print(panel)
        display(Image(filename=str(panel)))